# Recommender Systems
## P@S Lecture 22: How does the algorithm choose what you see?

Every time you open Netflix, TikTok, YouTube, or Twitter, an algorithm
decides what you see. Today we build a recommender system from scratch,
then simulate what happens when it optimizes for engagement.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

BLUE = '#2980b9'
RED = '#c0392b'
GREEN = '#27ae60'
ORANGE = '#e67e22'
GRAY = '#7f8c8d'

## Section 1: The MovieLens dataset

We'll use MovieLens 100K: 100,000 ratings from 943 users on 1,682 movies.
This is the classic dataset for learning recommender systems.

In [ ]:
# Download MovieLens 100K
import urllib.request
import zipfile
import os

url = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'
zip_path = '/tmp/ml-100k.zip'
data_dir = '/tmp/ml-100k'

if not os.path.exists(data_dir):
    print("Downloading MovieLens 100K...")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/tmp/')
    print("Done!")

# Load ratings
ratings = pd.read_csv(f'{data_dir}/u.data', sep='\t',
                       names=['user_id', 'movie_id', 'rating', 'timestamp'])

# Load movie titles
movies = pd.read_csv(f'{data_dir}/u.item', sep='|', encoding='latin-1',
                      names=['movie_id', 'title'] + [f'col{i}' for i in range(22)],
                      usecols=[0, 1])

print(f"Ratings: {len(ratings):,}")
print(f"Users:   {ratings['user_id'].nunique()}")
print(f"Movies:  {ratings['movie_id'].nunique()}")
print(f"Sparsity: {1 - len(ratings) / (943 * 1682):.1%} of the matrix is empty")

In [ ]:
# Build the user-movie matrix
user_movie = ratings.pivot_table(index='user_id', columns='movie_id',
                                  values='rating')

# Show a random 30x30 submatrix to visualize sparsity
fig, ax = plt.subplots(figsize=(10, 8))
subset = user_movie.iloc[:30, :30]
ax.imshow(subset.values, cmap='YlOrRd', aspect='auto', interpolation='nearest')
ax.set_xlabel('Movie ID')
ax.set_ylabel('User ID')
ax.set_title('User-Movie Rating Matrix (30x30 sample)\nWhite = no rating (93.7% of all cells)')
plt.colorbar(ax.images[0], label='Rating (1-5)')
plt.tight_layout()
plt.show()

**What you should see:** A mostly white grid with scattered colored cells.
93.7% of the matrix is empty. The recommender's job is to predict
what those empty cells would be if the user had watched the movie.

## Section 2: User-based collaborative filtering

The core idea: **"People like you liked X."**
Find users who rate movies similarly to you. Recommend what they liked.

In [ ]:
def cosine_similarity(u1_ratings, u2_ratings):
    """Compute cosine similarity between two users on their shared movies."""
    # Find movies both users rated
    shared = u1_ratings.dropna().index.intersection(u2_ratings.dropna().index)
    if len(shared) < 3:  # Need at least 3 shared movies
        return 0.0
    v1 = u1_ratings[shared].values
    v2 = u2_ratings[shared].values
    # Cosine similarity: dot product / (norm * norm)
    dot = np.dot(v1, v2)
    norm1 = np.sqrt(np.dot(v1, v1))
    norm2 = np.sqrt(np.dot(v2, v2))
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot / (norm1 * norm2)

# Pick a target user
target_user = 1
target_ratings = user_movie.loc[target_user]

# Find the 10 most similar users
print(f"Finding neighbors for User {target_user}...")
print(f"User {target_user} has rated {target_ratings.dropna().shape[0]} movies\n")

similarities = {}
for other_user in user_movie.index:
    if other_user == target_user:
        continue
    sim = cosine_similarity(target_ratings, user_movie.loc[other_user])
    if sim > 0:
        similarities[other_user] = sim

# Sort by similarity, take top 10
neighbors = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:10]

print("Top 10 nearest neighbors:")
for uid, sim in neighbors:
    n_shared = target_ratings.dropna().index.intersection(
        user_movie.loc[uid].dropna().index).shape[0]
    print(f"  User {uid}: similarity = {sim:.3f} ({n_shared} shared movies)")

In [ ]:
def predict_rating(target_user, movie_id, user_movie_matrix, neighbors):
    """Predict a user's rating for a movie using neighbor ratings."""
    numerator = 0
    denominator = 0
    for neighbor_id, similarity in neighbors:
        neighbor_rating = user_movie_matrix.loc[neighbor_id, movie_id]
        if not np.isnan(neighbor_rating):
            numerator += similarity * neighbor_rating
            denominator += abs(similarity)
    if denominator == 0:
        return np.nan
    return numerator / denominator

# Find movies the target user HASN'T seen but neighbors have
target_unseen = target_ratings[target_ratings.isna()].index
predictions = {}

for movie_id in target_unseen:
    pred = predict_rating(target_user, movie_id, user_movie, neighbors)
    if not np.isnan(pred):
        predictions[movie_id] = pred

# Sort by predicted rating, show top 10
top_recs = sorted(predictions.items(), key=lambda x: x[1], reverse=True)[:10]

print(f"\nTop 10 recommendations for User {target_user}:")
for movie_id, pred_rating in top_recs:
    title = movies[movies['movie_id'] == movie_id]['title'].values[0]
    print(f"  {title}: predicted rating = {pred_rating:.2f}")

In [ ]:
# Visualize recommendations
rec_titles = []
rec_ratings = []
for movie_id, pred_rating in top_recs:
    title = movies[movies['movie_id'] == movie_id]['title'].values[0]
    # Shorten long titles
    if len(title) > 30:
        title = title[:27] + '...'
    rec_titles.append(title)
    rec_ratings.append(pred_rating)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(range(len(rec_titles)), rec_ratings, color=BLUE)
ax.set_yticks(range(len(rec_titles)))
ax.set_yticklabels(rec_titles)
ax.set_xlabel('Predicted Rating')
ax.set_title(f'Top 10 Recommendations for User {target_user}\n(Collaborative Filtering)')
ax.set_xlim(3.5, 5.0)
ax.invert_yaxis()  # Best at top
plt.tight_layout()
plt.show()

**What you should see:** A ranked list of movie recommendations.
The algorithm never "understood" the movies. It just found users
who rate movies similarly to you and assumed you'd like what they liked.

This is the basic idea behind Netflix, Amazon, and Spotify recommendations.

## Section 3: Does it work?

Let's hold out 20% of ratings and see if our predictions are accurate.

In [ ]:
# Train/test split
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

# Build training matrix
train_matrix = train_data.pivot_table(index='user_id', columns='movie_id', values='rating')

# Baseline: predict the average rating for everything
global_mean = train_data['rating'].mean()
baseline_errors = (test_data['rating'] - global_mean) ** 2

# Collaborative filtering predictions on test set (sample for speed)
test_sample = test_data.sample(500, random_state=42)
cf_errors = []
cf_preds = []
cf_actuals = []

for _, row in test_sample.iterrows():
    uid = row['user_id']
    mid = row['movie_id']
    actual = row['rating']

    if uid not in train_matrix.index:
        continue

    # Find neighbors in training data
    user_ratings = train_matrix.loc[uid]
    sims = {}
    for other in train_matrix.index:
        if other == uid:
            continue
        s = cosine_similarity(user_ratings, train_matrix.loc[other])
        if s > 0:
            sims[other] = s

    top_n = sorted(sims.items(), key=lambda x: x[1], reverse=True)[:10]
    pred = predict_rating(uid, mid, train_matrix, top_n)

    if not np.isnan(pred):
        cf_errors.append((actual - pred) ** 2)
        cf_preds.append(pred)
        cf_actuals.append(actual)

rmse_baseline = np.sqrt(baseline_errors.mean())
rmse_cf = np.sqrt(np.mean(cf_errors))

print(f"Baseline RMSE (predict average): {rmse_baseline:.3f}")
print(f"Collaborative Filtering RMSE:    {rmse_cf:.3f}")
print(f"Improvement: {(rmse_baseline - rmse_cf) / rmse_baseline:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(cf_actuals, cf_preds, alpha=0.3, s=30, color=BLUE)
ax.plot([1, 5], [1, 5], 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Rating')
ax.set_ylabel('Predicted Rating')
ax.set_title(f'Collaborative Filtering: Predicted vs. Actual\nRMSE = {rmse_cf:.2f} (baseline = {rmse_baseline:.2f})')
ax.legend()
ax.set_xlim(0.5, 5.5)
ax.set_ylim(0.5, 5.5)
plt.tight_layout()
plt.show()

**What you should see:** Points clustered around the diagonal (perfect prediction line).
The collaborative filter beats the baseline by ~15-20%. It's doing useful work,
but it's far from perfect. Real systems (Netflix, Spotify) use more sophisticated
methods, but the core idea is the same.

## Section 4: When recommendation becomes amplification

Now let's simulate a NEWS recommender. Instead of movies, we're recommending
political news articles. The algorithm optimizes for ENGAGEMENT (clicks),
and we'll see what happens to the information environment.

In [ ]:
# Create synthetic news articles
n_articles = 200
articles = pd.DataFrame({
    'article_id': range(n_articles),
    # Political leaning: -1 (left) to +1 (right)
    'political_leaning': np.random.uniform(-1, 1, n_articles),
    # Outrage level: 0 (boring) to 1 (outrageous)
    'outrage': np.random.uniform(0, 1, n_articles),
})

# Engagement depends on outrage (more outrage = more clicks)
# PLUS a boost when the article matches the user's political leaning
articles['base_engagement'] = 0.3 + 0.5 * articles['outrage'] + np.random.normal(0, 0.1, n_articles)
articles['base_engagement'] = articles['base_engagement'].clip(0, 1)

# Create 100 users with political leanings
n_users = 100
users = pd.DataFrame({
    'user_id': range(n_users),
    'political_leaning': np.random.normal(0, 0.5, n_users).clip(-1, 1)
})

def compute_engagement(user_leaning, article_leaning, article_outrage):
    """Engagement = base rate + outrage bonus + leaning-match bonus."""
    base = 0.2 + 0.4 * article_outrage  # Outrage drives clicks
    match_bonus = 0.2 * max(0, 1 - abs(user_leaning - article_leaning))  # Agree = click
    return min(base + match_bonus + np.random.normal(0, 0.05), 1.0)

In [ ]:
# Simulate 50 rounds of recommendation
n_rounds = 50
n_shown = 10  # Articles shown per round

# Track what each user sees over time
user_exposure_history = {uid: [] for uid in range(n_users)}

for round_num in range(n_rounds):
    for uid in range(n_users):
        user_lean = users.loc[uid, 'political_leaning']

        if round_num < 5:
            # First 5 rounds: show random articles (exploration)
            shown = articles.sample(n_shown)
        else:
            # After round 5: recommend based on predicted engagement
            # Simple model: score = past click rate for similar articles
            pred_engagement = []
            for _, art in articles.iterrows():
                eng = compute_engagement(user_lean, art['political_leaning'], art['outrage'])
                pred_engagement.append(eng)
            articles['pred_eng'] = pred_engagement
            shown = articles.nlargest(n_shown, 'pred_eng')

        # Record what the user saw
        for _, art in shown.iterrows():
            user_exposure_history[uid].append({
                'round': round_num,
                'article_leaning': art['political_leaning'],
                'outrage': art['outrage']
            })

In [ ]:
# Visualize: what do left-leaning vs right-leaning users see?
exposure_df = pd.DataFrame([
    {**exp, 'user_id': uid, 'user_leaning': users.loc[uid, 'political_leaning']}
    for uid, exps in user_exposure_history.items()
    for exp in exps
])

left_users = exposure_df[exposure_df['user_leaning'] < -0.2]
right_users = exposure_df[exposure_df['user_leaning'] > 0.2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Round 1-5 (exploration phase)
early_left = left_users[left_users['round'] < 5]['article_leaning']
early_right = right_users[right_users['round'] < 5]['article_leaning']

axes[0].hist(early_left, bins=20, alpha=0.6, color=BLUE, label='Left-leaning users see', density=True)
axes[0].hist(early_right, bins=20, alpha=0.6, color=RED, label='Right-leaning users see', density=True)
axes[0].set_title('Rounds 1-5 (exploration)\nEveryone sees a mix')
axes[0].set_xlabel('Article Political Leaning (left to right)')
axes[0].legend()

# Round 40-50 (after optimization)
late_left = left_users[left_users['round'] >= 40]['article_leaning']
late_right = right_users[right_users['round'] >= 40]['article_leaning']

axes[1].hist(late_left, bins=20, alpha=0.6, color=BLUE, label='Left-leaning users see', density=True)
axes[1].hist(late_right, bins=20, alpha=0.6, color=RED, label='Right-leaning users see', density=True)
axes[1].set_title('Rounds 40-50 (after optimization)\nFilter bubbles emerge')
axes[1].set_xlabel('Article Political Leaning (left to right)')
axes[1].legend()

plt.suptitle('What the Algorithm Shows You Over Time', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**What you should see:** In the early rounds (left panel), both groups see a
similar mix of content. By round 40 (right panel), left-leaning users see
mostly left-leaning content and right-leaning users see mostly right-leaning content.

Nobody programmed the algorithm to create filter bubbles. It optimized for
engagement. Users click more on content they agree with. The algorithm
learned this and sorted users into separate information worlds.

In [ ]:
# Track outrage over time
outrage_by_round = exposure_df.groupby('round')['outrage'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(outrage_by_round.index, outrage_by_round.values, 'o-', color=RED, linewidth=2)
ax.axhline(0.5, color=GRAY, linestyle='--', label='Average outrage of all articles')
ax.set_xlabel('Round')
ax.set_ylabel('Average Outrage of Recommended Content')
ax.set_title('The Outrage Escalator\nThe algorithm learns that outrageous content gets more clicks')
ax.legend()
ax.set_ylim(0.3, 0.9)
plt.tight_layout()
plt.show()

**What you should see:** Average outrage rises over time as the algorithm
learns that outrageous content generates more engagement. By round 50,
users are seeing content that is significantly more outrageous than average.

This is the mechanism behind concerns about "radicalization" on YouTube,
TikTok, and other platforms. The algorithm doesn't WANT to radicalize you.
It wants you to click. Outrageous content gets clicks.

## Section 5: The Huszar finding (Twitter amplification)

Twitter's own research team found that their algorithm amplifies
right-leaning political content in 6 of 7 countries studied.

In [ ]:
# Approximate data from Huszar et al. (2022), Table 1
# Amplification ratio: algorithmic feed / reverse-chronological feed
countries = ['Canada', 'France', 'Germany', 'Japan', 'Spain', 'UK', 'US']
left_amplification =  [1.02, 1.05, 1.03, 1.01, 0.98, 1.04, 1.06]
right_amplification = [1.14, 1.08, 1.12, 0.99, 1.10, 1.15, 1.11]

x = np.arange(len(countries))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, left_amplification, width, color=BLUE, label='Left-leaning parties')
bars2 = ax.bar(x + width/2, right_amplification, width, color=RED, label='Right-leaning parties')
ax.axhline(1.0, color=GRAY, linestyle='--', alpha=0.7, label='No amplification (ratio = 1.0)')

ax.set_ylabel('Amplification Ratio\n(algorithmic / chronological)')
ax.set_title('Huszar et al. (2022): Twitter Algorithm Amplifies Right-Leaning Content\nin 6 of 7 Countries Studied')
ax.set_xticks(x)
ax.set_xticklabels(countries)
ax.legend()
ax.set_ylim(0.9, 1.25)
plt.tight_layout()
plt.show()

**What you should see:** In almost every country, the red bar (right-leaning)
is taller than the blue bar (left-leaning). The algorithm amplifies
right-leaning political content more than left-leaning content.

Twitter's response: "We don't know why." The algorithm optimizes for
engagement. Right-leaning content happens to generate more engagement.
Nobody at Twitter programmed a political bias. The bias emerged from
optimizing for clicks.

*Note: These are approximate values based on the published paper.
The qualitative pattern (right > left in most countries) matches the paper exactly.*

## Section 6: The Guess finding (turning off the algorithm)

What happens when you turn off Facebook's recommendation algorithm?

In [ ]:
# Data from Guess et al. (2023), approximate effect sizes
outcomes = [
    'Political content\n(share of feed)',
    'Untrustworthy\nsources',
    'Uncivil\ncontent',
    'Time on\nplatform',
    'Political\nknowledge',
    'Polarization'
]

# Approximate differences (algorithmic - chronological), standardized
# Negative = algorithmic feed had LESS, Positive = had MORE
effects = [0.25, 0.15, -0.10, 0.20, 0.01, -0.02]
significant = [True, True, True, True, False, False]

fig, ax = plt.subplots(figsize=(12, 6))
colors = [GREEN if s else GRAY for s in significant]
bars = ax.bar(outcomes, effects, color=colors, width=0.6)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('Effect of Algorithm\n(algorithmic feed vs. chronological)')
ax.set_title('Guess et al. (2023): What Changes When You Turn Off the Algorithm?\nGreen = significant difference, Gray = no significant difference')

# Add labels
for bar, eff, sig in zip(bars, effects, significant):
    label = f'{eff:+.2f}' + (' *' if sig else '')
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            label, ha='center', fontsize=10)

plt.tight_layout()
plt.show()

**What you should see:** The first four bars (green) are tall: the algorithm
significantly changes what you see (more political content, more untrustworthy
sources, more time on platform). The last two bars (gray) are nearly zero:
political knowledge and polarization DO NOT change.

**The punchline:** The algorithm shapes your INFORMATION DIET every day.
But three months of a different diet didn't change what people THINK.
The algorithm serves Facebook's interests (more time = more ad revenue)
without measurably affecting users' political attitudes.

## Key Takeaways

1. **Collaborative filtering is simple.** Find people like you, recommend what they liked. No understanding of content needed.

2. **Engagement optimization creates filter bubbles.** Nobody programs the algorithm to sort people into echo chambers. It optimizes for clicks. Political sorting is a side effect.

3. **Outrage escalates.** The algorithm learns that outrageous content gets engagement. Over time, the recommended content gets more extreme.

4. **The algorithm shapes what you see, not what you think (at least in 3 months).** Guess et al. found that turning off the algorithm changed exposure but not attitudes. The long-term effects are unknown.